In [ ]:
# ==============================================================================
# File: 02_Train_ZeroFake.ipynb (Cell 1 of 7)
# What this notebook does: Mounts Google Drive, clones/pulls https://github.com/Alekhagrawal/ZeroFake.git,
#   unzips cropped_faces.zip onto Colab's fast local SSD (/content/data), and runs all four
#   ZeroFake ablation training experiments (Baseline, Contrastive, Single-Prototype, and Full ZeroFake)
#   plus the optional second Leave-One-Generator-Out rotation holding out FaceSwap.
# Inputs: /content/drive/MyDrive/ZeroFake_Storage/cropped_faces.zip and GitHub repo.
# Outputs: Saved last_checkpoint.pt and best_model.pt (including calibrated UNKNOWN
#   threshold tau_reject) inside /content/drive/MyDrive/ZeroFake_Storage/checkpoints/.
# ==============================================================================

# --- CELL 1: Mount Google Drive & Verify cropped_faces.zip Exists ---
import os
import shutil
from google.colab import drive

# 1. Safely clear any unmounted placeholder /content/drive folder before mounting Google Drive
if not os.path.ismount('/content/drive'):
    shutil.rmtree('/content/drive', ignore_errors=True)
drive.mount('/content/drive')

STORAGE_DIR = '/content/drive/MyDrive/ZeroFake_Storage'
ZIP_PATH = os.path.join(STORAGE_DIR, 'cropped_faces.zip')

# 2. Verify that cropped_faces.zip from 01_Prepare_Data.ipynb exists
if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        f'cropped_faces.zip not found at {ZIP_PATH}! '
        'Please run 01_Prepare_Data.ipynb first to prepare and back up the dataset.'
    )

zip_size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
print(f'[SUCCESS] Found cropped_faces.zip on Google Drive ({zip_size_mb:.2f} MB): {ZIP_PATH}')


In [ ]:
# --- CELL 2: Clone or Force-Update Latest Code from GitHub & Install Dependencies ---
# Plain-English explanation:
#   Clones https://github.com/Alekhagrawal/ZeroFake.git into /content/ZeroFake (or fetches
#   and resets to origin/main if already cloned so no stale code ever remains in Colab),
#   installs facenet-pytorch with --no-deps first (avoiding Pillow 10.2.0 compilation on Python 3.13),
#   installs remaining packages using 'grad-cam', and runs a fast --dry-run check.

import os
import sys

GITHUB_REPO_URL = 'https://github.com/Alekhagrawal/ZeroFake.git'
REPO_DIR = '/content/ZeroFake'

if os.path.exists(os.path.join(REPO_DIR, '.git')):
    print(f'[INFO] Fetching and syncing latest commits from {GITHUB_REPO_URL}...')
    !git -C /content/ZeroFake fetch origin
    !git -C /content/ZeroFake reset --hard origin/main
else:
    print(f'[INFO] Cloning {GITHUB_REPO_URL} into {REPO_DIR}...')
    !rm -rf /content/ZeroFake
    !git clone https://github.com/Alekhagrawal/ZeroFake.git /content/ZeroFake

%cd /content/ZeroFake
os.environ['PYTHONPATH'] = '/content/ZeroFake'
if '/content/ZeroFake' not in sys.path:
    sys.path.insert(0, '/content/ZeroFake')

# Ensure datasets/__init__.py exists so Colab's Hugging Face 'datasets' package never shadows local datasets/
os.makedirs('/content/ZeroFake/datasets', exist_ok=True)
init_file = '/content/ZeroFake/datasets/__init__.py'
if not os.path.exists(init_file):
    with open(init_file, 'w', encoding='utf-8') as f:
        f.write('from datasets.deepfake_dataset import DeepfakeDataset, GeneratorBalancedBatchSampler\n')

# 1. Install facenet-pytorch separately with --no-deps so pip does not compile Pillow 10.2.0 on Python 3.13
!pip install -q --no-deps facenet-pytorch

# 2. Install remaining packages using the official PyPI package name 'grad-cam'
!pip install -q timm opencv-python-headless scikit-learn matplotlib seaborn grad-cam gradio pyyaml tqdm

# 3. Verify architecture & training engine with a 1-second CPU dry-run before starting GPU training
!PYTHONPATH=/content/ZeroFake python -u training/train.py --dry-run
if _exit_code != 0:
    raise RuntimeError(f'Pre-flight --dry-run failed with exit code {_exit_code}')
print('[SUCCESS] Repository synced at /content/ZeroFake, dependencies installed, and --dry-run verified!')


In [ ]:
# --- CELL 3: Unzip cropped_faces.zip onto Colab's Fast Local SSD (/content/data) ---
# Plain-English explanation:
#   Reading 60,000 individual image files directly across Google Drive causes severe
#   network latency. Unzipping cropped_faces.zip once onto Colab's local /content/data
#   SSD lets the L4/A100 GPU train at maximum speed (~20-35 minutes per 15-epoch run).

import os
import shutil
import zipfile

STORAGE_DIR = '/content/drive/MyDrive/ZeroFake_Storage'
ZIP_PATH = os.path.join(STORAGE_DIR, 'cropped_faces.zip')
LOCAL_DATA_DIR = '/content/data'
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

crops_dir = os.path.join(LOCAL_DATA_DIR, 'crops')
if not os.path.exists(crops_dir):
    print(f'[INFO] Unzipping {ZIP_PATH} onto local SSD {LOCAL_DATA_DIR}...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(LOCAL_DATA_DIR)
    print('[SUCCESS] Extracted all cropped faces onto local SSD (/content/data).')
else:
    print('[INFO] Cropped faces already extracted in /content/data/crops.')

# Copy splits.json and frames_manifest.csv from Google Drive into /content/data if not already inside zip
for fname in ('splits.json', 'frames_manifest.csv', 'failed_detections.csv'):
    drive_file = os.path.join(STORAGE_DIR, fname)
    local_file = os.path.join(LOCAL_DATA_DIR, fname)
    if os.path.exists(drive_file) and not os.path.exists(local_file):
        shutil.copy2(drive_file, local_file)

# Handle nested processed_faces/ directory if zip archive included top-level folder name
nested_dir = os.path.join(LOCAL_DATA_DIR, 'processed_faces')
if os.path.exists(nested_dir) and not os.path.exists(os.path.join(LOCAL_DATA_DIR, 'frames_manifest.csv')):
    for item in os.listdir(nested_dir):
        src_item = os.path.join(nested_dir, item)
        dst_item = os.path.join(LOCAL_DATA_DIR, item)
        if not os.path.exists(dst_item):
            shutil.move(src_item, dst_item)

assert os.path.exists(os.path.join(LOCAL_DATA_DIR, 'frames_manifest.csv')), (
    'frames_manifest.csv not found in /content/data! Check ZeroFake_Storage on Google Drive.'
)
print(f'[READY] Local SSD dataset directory contents: {os.listdir(LOCAL_DATA_DIR)}')


In [ ]:
# --- CELL 4: Train Config A (Baseline: --mode baseline) for 15 Epochs ---
# Plain-English explanation:
#   Trains the standard ConvNeXt-Tiny binary classifier baseline (with no contrastive
#   loss or prototypes) for 15 epochs. Saves last_checkpoint.pt and best_model.pt
#   after every epoch to /content/drive/MyDrive/ZeroFake_Storage/checkpoints/baseline/
#   and automatically resumes if a previous run was interrupted.

import os

%cd /content/ZeroFake
BASELINE_SAVE_DIR = '/content/drive/MyDrive/ZeroFake_Storage/checkpoints/baseline'
os.makedirs(BASELINE_SAVE_DIR, exist_ok=True)

print('[TRAIN] Starting Run 1 — Config A (Baseline: --mode baseline) for 15 epochs...')
!PYTHONPATH=/content/ZeroFake python -u training/train.py --mode baseline --epochs 15 --batch_size 64 --lr 1e-4 --data_dir /content/data --save_dir /content/drive/MyDrive/ZeroFake_Storage/checkpoints/baseline --resume auto
if _exit_code != 0:
    raise RuntimeError(f'Config A (Baseline) training failed with exit code {_exit_code}')
print(f'[SUCCESS] Config A (Baseline) checkpoints saved to: {BASELINE_SAVE_DIR}')


In [ ]:
# --- CELL 5: Train Config B (--mode contrastive) & Config C (--mode single_proto) for 15 Epochs Each ---
# Plain-English explanation:
#   - Run 2 (Config B — contrastive): Adds Multi-Generator Supervised Contrastive Loss (L_con)
#     to align fake features across generators and cluster real faces tightly.
#   - Run 3 (Config C — single_proto): Adds 1 Real prototype and 1 Fake prototype (K=1) to
#     test the single-cluster fake assumption before upgrading to K=6 diverse Fake prototypes.
#   Both runs save checkpoints after every epoch to their dedicated Google Drive folders.

import os

%cd /content/ZeroFake
CONTRASTIVE_SAVE_DIR = '/content/drive/MyDrive/ZeroFake_Storage/checkpoints/contrastive'
SINGLE_PROTO_SAVE_DIR = '/content/drive/MyDrive/ZeroFake_Storage/checkpoints/single_proto'
os.makedirs(CONTRASTIVE_SAVE_DIR, exist_ok=True)
os.makedirs(SINGLE_PROTO_SAVE_DIR, exist_ok=True)

# Run 2 — Config B: Contrastive
print('[TRAIN] Starting Run 2 — Config B (Contrastive: --mode contrastive) for 15 epochs...')
!PYTHONPATH=/content/ZeroFake python -u training/train.py --mode contrastive --epochs 15 --batch_size 64 --lr 1e-4 --data_dir /content/data --save_dir /content/drive/MyDrive/ZeroFake_Storage/checkpoints/contrastive --resume auto
if _exit_code != 0:
    raise RuntimeError(f'Config B (Contrastive) training failed with exit code {_exit_code}')
print(f'[SUCCESS] Config B (Contrastive) checkpoints saved to: {CONTRASTIVE_SAVE_DIR}')

# Run 3 — Config C: Single Prototype (1 Real + K=1 Fake)
print('[TRAIN] Starting Run 3 — Config C (Single Prototype: --mode single_proto) for 15 epochs...')
!PYTHONPATH=/content/ZeroFake python -u training/train.py --mode single_proto --epochs 15 --batch_size 64 --lr 1e-4 --data_dir /content/data --save_dir /content/drive/MyDrive/ZeroFake_Storage/checkpoints/single_proto --resume auto
if _exit_code != 0:
    raise RuntimeError(f'Config C (Single Prototype) training failed with exit code {_exit_code}')
print(f'[SUCCESS] Config C (Single Prototype) checkpoints saved to: {SINGLE_PROTO_SAVE_DIR}')


In [ ]:
# --- CELL 6: Train Full ZeroFake (Config D/E: --mode zerofake) for 15 Epochs ---
# Plain-English explanation:
#   Trains Run 4 (Config D/E — Full ZeroFake) combining ConvNeXt-Tiny, Multi-Generator
#   Supervised Contrastive Learning (L_con), 1 Real + K=6 diverse Fake prototypes (L_proto),
#   and validation-calibrated Open-Set Uncertainty U(x). Saves best_model.pt (with the
#   calibrated UNKNOWN threshold tau_reject locked in at 95% validation coverage) to
#   /content/drive/MyDrive/ZeroFake_Storage/checkpoints/zerofake/.

import os

%cd /content/ZeroFake
ZEROFAKE_SAVE_DIR = '/content/drive/MyDrive/ZeroFake_Storage/checkpoints/zerofake'
os.makedirs(ZEROFAKE_SAVE_DIR, exist_ok=True)

print('[TRAIN] Starting Run 4 — Config D/E (Full ZeroFake: --mode zerofake) for 15 epochs...')
!PYTHONPATH=/content/ZeroFake python -u training/train.py --mode zerofake --epochs 15 --batch_size 64 --lr 1e-4 --data_dir /content/data --save_dir /content/drive/MyDrive/ZeroFake_Storage/checkpoints/zerofake --resume auto
if _exit_code != 0:
    raise RuntimeError(f'Full ZeroFake (Config D/E) training failed with exit code {_exit_code}')
print(f'[SUCCESS] Full ZeroFake best_model.pt (with calibrated UNKNOWN threshold) saved to: {ZEROFAKE_SAVE_DIR}')


In [ ]:
# --- CELL 7: Optional Second LOGO Rotation Holding Out 'FaceSwap' (Zero Re-Cropping) ---
# Plain-English explanation:
#   1. Builds /content/data/splits_holdout_faceswap.json directly from /content/data/splits.json
#      and /content/data/frames_manifest.csv without re-cropping any face images.
#   2. Preserves the exact same connected-component actor groups (e.g., 000_003.mp4) so zero
#      actor identities leak across train (80%), val (10%), and test_closed (10%), while
#      holding out 'FaceSwap' 100% in test_unseen and placing 'NeuralTextures' into each
#      actor's existing identity_partition.
#   3. Trains both --mode baseline and --mode zerofake for 15 epochs each using
#      --splits_path /content/data/splits_holdout_faceswap.json, saving checkpoints to
#      /content/drive/MyDrive/ZeroFake_Storage/checkpoints/holdout_faceswap/{baseline,zerofake}.

import csv
import json
import os
import sys

%cd /content/ZeroFake
if '/content/ZeroFake' not in sys.path:
    sys.path.insert(0, '/content/ZeroFake')

from preprocessing.create_splits import create_logo_splits, save_splits_json, verify_splits

DATA_DIR = '/content/data'
ORIG_SPLITS_PATH = os.path.join(DATA_DIR, 'splits.json')
MANIFEST_PATH = os.path.join(DATA_DIR, 'frames_manifest.csv')
FACESWAP_SPLITS_PATH = os.path.join(DATA_DIR, 'splits_holdout_faceswap.json')

# 1. Reconstruct all unique video records from splits.json (and frames_manifest.csv) preserving
#    existing connected-component identity_partition assignments ('train', 'val', 'test_closed').
if os.path.exists(ORIG_SPLITS_PATH):
    with open(ORIG_SPLITS_PATH, 'r', encoding='utf-8') as f:
        orig_payload = json.load(f)

    all_video_records = []
    for sp_list in orig_payload.get('splits', {}).values():
        all_video_records.extend(sp_list)

    new_splits = {'train': [], 'val': [], 'test_closed': [], 'test_unseen': []}
    for rec in all_video_records:
        rec_copy = dict(rec)
        actor_partition = rec_copy.get('identity_partition', rec_copy.get('split', 'train'))
        if actor_partition == 'test_unseen':
            new_splits = None
            break
        if rec_copy['generator_name'] == 'FaceSwap':
            rec_copy['split'] = 'test_unseen'
            new_splits['test_unseen'].append(rec_copy)
        else:
            rec_copy['split'] = actor_partition
            new_splits[actor_partition].append(rec_copy)

    if new_splits is not None:
        faceswap_payload = {
            'metadata': {
                'holdout_generator': 'FaceSwap',
                'seen_generators': ['Deepfakes', 'Face2Face', 'NeuralTextures'],
                'seed': orig_payload.get('metadata', {}).get('seed', 42),
                'num_connected_identity_components': orig_payload.get('metadata', {}).get('num_connected_identity_components', 0),
                'counts': {k: len(v) for k, v in new_splits.items()},
            },
            'splits': new_splits,
        }
        verify_splits(faceswap_payload)
    else:
        faceswap_payload = create_logo_splits(all_video_records, holdout_generator='FaceSwap', seed=42)
else:
    # Fallback: build unique video records directly from frames_manifest.csv
    seen_vids = {}
    with open(MANIFEST_PATH, 'r', encoding='utf-8') as mf:
        for row in csv.DictReader(mf):
            vid = row['video_id'].strip()
            if vid not in seen_vids:
                ids = [x for x in row.get('identity_ids', '').split('|') if x]
                seen_vids[vid] = {
                    'video_id': vid,
                    'video_path': f"{row['generator_name']}/{vid.split('/')[-1]}.mp4",
                    'filename': f"{vid.split('/')[-1]}.mp4",
                    'label': int(row['label']),
                    'generator_name': row['generator_name'].strip(),
                    'identity_ids': ids if ids else [vid.split('/')[-1]],
                }
    faceswap_payload = create_logo_splits(list(seen_vids.values()), holdout_generator='FaceSwap', seed=42)

save_splits_json(faceswap_payload, FACESWAP_SPLITS_PATH)
print(f"[SUCCESS] Saved verified FaceSwap LOGO splits to {FACESWAP_SPLITS_PATH}: {faceswap_payload['metadata']['counts']}")

# 2. Train Baseline and Full ZeroFake for 15 epochs holding out FaceSwap
FS_BASELINE_DIR = '/content/drive/MyDrive/ZeroFake_Storage/checkpoints/holdout_faceswap/baseline'
FS_ZEROFAKE_DIR = '/content/drive/MyDrive/ZeroFake_Storage/checkpoints/holdout_faceswap/zerofake'
os.makedirs(FS_BASELINE_DIR, exist_ok=True)
os.makedirs(FS_ZEROFAKE_DIR, exist_ok=True)

print('[TRAIN] Starting FaceSwap-Holdout Baseline (--mode baseline) for 15 epochs...')
!PYTHONPATH=/content/ZeroFake python -u training/train.py --mode baseline --holdout_generator FaceSwap --epochs 15 --batch_size 64 --lr 1e-4 --data_dir /content/data --splits_path /content/data/splits_holdout_faceswap.json --save_dir /content/drive/MyDrive/ZeroFake_Storage/checkpoints/holdout_faceswap/baseline --resume auto
if _exit_code != 0:
    raise RuntimeError(f'FaceSwap-Holdout Baseline training failed with exit code {_exit_code}')
print(f'[SUCCESS] FaceSwap-Holdout Baseline checkpoints saved to: {FS_BASELINE_DIR}')

print('[TRAIN] Starting FaceSwap-Holdout Full ZeroFake (--mode zerofake) for 15 epochs...')
!PYTHONPATH=/content/ZeroFake python -u training/train.py --mode zerofake --holdout_generator FaceSwap --epochs 15 --batch_size 64 --lr 1e-4 --data_dir /content/data --splits_path /content/data/splits_holdout_faceswap.json --save_dir /content/drive/MyDrive/ZeroFake_Storage/checkpoints/holdout_faceswap/zerofake --resume auto
if _exit_code != 0:
    raise RuntimeError(f'FaceSwap-Holdout Full ZeroFake training failed with exit code {_exit_code}')
print(f'[SUCCESS] FaceSwap-Holdout Full ZeroFake checkpoints saved to: {FS_ZEROFAKE_DIR}')
